# Daily Challenge: Building Trustworthy Insights with BERT
## Week 7 — Day 5 | DI GenAI & Machine Learning Bootcamp 2026

**Scenario:** Deploy a trustworthy AI assistant for customer support — one that surfaces not just a prediction but the evidence tokens behind it.

**Tasks:**
1. Data Loading & Inspection — `tweet_eval` sentiment (3 classes)
2. Tokenization Pipeline — DistilBERT tokenizer, 128-token sequences
3. Fine-Tuning — `AutoModelForSequenceClassification`, Trainer API
4. Evaluation & Calibration — accuracy, F1, softmax confidence histogram
5. Attention Inspection — CLS attention heatmap + `analyze_text()` helper

> **Run on Google Colab with GPU** — DistilBERT trains in ~10 min on T4.

In [ ]:
!pip install --quiet transformers datasets evaluate accelerate seaborn matplotlib torch

In [ ]:
import os
import warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    TrainingArguments,
    Trainer,
)
import evaluate
warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} ✓  |  Device: {DEVICE}")

LABEL_NAMES = {0: "negative", 1: "neutral", 2: "positive"}
MODEL_NAME   = "distilbert-base-uncased"
MAX_LEN      = 128
OUTPUT_DIR   = "./distilbert_tweet_eval"

## Task 1 — Data Loading & Inspection

`tweet_eval` sentiment uses 3 classes: **0 = negative · 1 = neutral · 2 = positive**.

In [ ]:
raw = load_dataset("tweet_eval", "sentiment")

print("Dataset splits:")
for split, ds in raw.items():
    counts = Counter(ds["label"])
    dist   = "  ".join(f"{LABEL_NAMES[k]}={v:,}" for k, v in sorted(counts.items()))
    print(f"  {split:<12} {len(ds):>6,} examples  |  {dist}")

print(f"\nLabels: {LABEL_NAMES}")

# ── Save 2 examples per label for later visualization ─────────────────────────
examples_by_label = {lbl: [] for lbl in LABEL_NAMES}
for item in raw["test"]:
    lbl = item["label"]
    if len(examples_by_label[lbl]) < 2:
        examples_by_label[lbl].append(item["text"])
    if all(len(v) >= 2 for v in examples_by_label.values()):
        break

print("\n── 2 examples per label (from test split) ──")
for lbl, texts in examples_by_label.items():
    print(f"\n  [{LABEL_NAMES[lbl].upper()}]")
    for t in texts:
        print(f"    • {t[:120]}")

# ── Class distribution bar chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
COLORS = {"negative": "#d9534f", "neutral": "#f0ad4e", "positive": "#5cb85c"}

for ax, (split, ds) in zip(axes, raw.items()):
    counts = Counter(ds["label"])
    names  = [LABEL_NAMES[k] for k in sorted(counts)]
    vals   = [counts[k]      for k in sorted(counts)]
    bars   = ax.bar(names, vals, color=[COLORS[n] for n in names], edgecolor="white")
    ax.set_title(f"{split.capitalize()} split", fontweight="bold")
    ax.set_ylabel("Tweets")
    ax.grid(True, alpha=0.3, axis="y")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f"{v:,}", ha="center", fontsize=9)

plt.suptitle("tweet_eval Sentiment — Class Distribution", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved ✓")

## Task 2 — Tokenization Pipeline

`distilbert-base-uncased` expects sequences padded / truncated to a fixed length. We use `max_length=128` which covers the vast majority of tweets (140-character limit means most fit under 50 tokens).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer : {MODEL_NAME}  (vocab={tokenizer.vocab_size:,})")


def preprocess(batch):
    """Tokenize a batch and keep the label column."""
    encoded = tokenizer(
        batch["text"],
        truncation = True,
        padding    = "max_length",
        max_length = MAX_LEN,
    )
    encoded["labels"] = batch["label"]
    return encoded


tok_ds = raw.map(preprocess, batched=True, remove_columns=["text"])
tok_ds.set_format("torch")

print(f"\nTokenized dataset:")
for split, ds in tok_ds.items():
    print(f"  {split:<12} {len(ds):,} examples  |  columns: {ds.column_names}")

# Quick sanity-check on one example
sample = tok_ds["train"][0]
print(f"\nFirst train example:")
print(f"  input_ids     shape : {sample['input_ids'].shape}")
print(f"  attention_mask shape: {sample['attention_mask'].shape}")
print(f"  label               : {sample['labels'].item()}  ({LABEL_NAMES[sample['labels'].item()]})")
print(f"  real tokens (mask=1): {int(sample['attention_mask'].sum())}")

## Task 3 — Fine-Tuning with the Trainer API

Key choices:
- **`load_best_model_at_end=True`** + **`metric_for_best_model="f1"`** — automatically restores the checkpoint with highest macro F1 after training
- **`weight_decay=0.01`** — L2 regularisation to prevent overfitting on the relatively small training split
- **`lr=5e-5`** — standard upper bound for DistilBERT fine-tuning; higher risks destabilising pretrained weights

In [ ]:
clf_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 3,
    id2label   = {v: k for v, k in enumerate(["negative", "neutral", "positive"])},
    label2id   = {"negative": 0, "neutral": 1, "positive": 2},
)

n_params = sum(p.numel() for p in clf_model.parameters())
print(f"Model loaded  : {MODEL_NAME}  ({n_params:,} parameters)")

# ── Metrics ───────────────────────────────────────────────────────────────────
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1    = f1_metric.compute(predictions=preds,  references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

# ── Training arguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate               = 5e-5,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    report_to                   = "none",
    logging_steps               = 50,
)

trainer = Trainer(
    model           = clf_model,
    args            = training_args,
    train_dataset   = tok_ds["train"],
    eval_dataset    = tok_ds["validation"],
    compute_metrics = compute_metrics,
)

print(f"\nTraining  : {len(tok_ds['train']):,} examples")
print(f"Validation: {len(tok_ds['validation']):,} examples")
print(f"\nStarting training…")
train_result = trainer.train()
print("\nTraining complete ✓")
print(f"  Best checkpoint : {trainer.state.best_model_checkpoint}")

In [ ]:
# ── Training curves from log history ─────────────────────────────────────────
log_history = trainer.state.log_history

train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
eval_logs  = [l for l in log_history if "eval_loss" in l]

if train_logs and eval_logs:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Loss
    axes[0].plot([l["step"] for l in train_logs], [l["loss"] for l in train_logs],
                 "b-", alpha=0.6, label="Train (step)")
    axes[0].plot([l["epoch"] * len(tok_ds["train"]) // 32
                  for l in eval_logs],
                 [l["eval_loss"] for l in eval_logs],
                 "r-o", label="Val (epoch)")
    axes[0].set_title("Loss", fontweight="bold"); axes[0].legend()
    axes[0].set_xlabel("Step"); axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(range(1, len(eval_logs)+1), [l["eval_accuracy"] for l in eval_logs],
                 "b-o", label="Val Accuracy")
    axes[1].set_title("Accuracy", fontweight="bold"); axes[1].legend()
    axes[1].set_xlabel("Epoch"); axes[1].grid(True, alpha=0.3)

    # F1
    axes[2].plot(range(1, len(eval_logs)+1), [l["eval_f1"] for l in eval_logs],
                 "g-o", label="Val Macro F1")
    axes[2].set_title("Macro F1", fontweight="bold"); axes[2].legend()
    axes[2].set_xlabel("Epoch"); axes[2].grid(True, alpha=0.3)

    plt.suptitle("DistilBERT Fine-Tuning — tweet_eval Sentiment", fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Plot saved ✓")

# Save best model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nBest model saved → {OUTPUT_DIR}")

## Task 4 — Evaluation & Calibration

We evaluate on the **validation** split (metrics logged during training) then collect **softmax confidence scores** on the **test** split to check for over/under-confidence.

A well-calibrated model should have most high-confidence predictions be correct.  
A spike near 1.0 with many wrong predictions signals **overconfidence**.

In [ ]:
# ── Validation evaluation ────────────────────────────────────────────────────
val_results = trainer.evaluate(tok_ds["validation"])
print("Validation metrics:")
print(f"  Accuracy   : {val_results['eval_accuracy']*100:.2f}%")
print(f"  Macro F1   : {val_results['eval_f1']*100:.2f}%")
print(f"  Loss       : {val_results['eval_loss']:.4f}")

# ── Test-set confidence collection ───────────────────────────────────────────
clf_model.eval()
clf_model.to(DEVICE)

all_confidences, all_preds, all_labels_list = [], [], []

for i in range(0, len(tok_ds["test"]), 64):
    batch = tok_ds["test"][i : i + 64]
    ids   = batch["input_ids"].to(DEVICE)
    mask  = batch["attention_mask"].to(DEVICE)
    labs  = batch["labels"].tolist()

    with torch.no_grad():
        logits = clf_model(input_ids=ids, attention_mask=mask).logits
    probs = F.softmax(logits, dim=-1).cpu().numpy()

    preds       = probs.argmax(axis=-1).tolist()
    confidences = probs.max(axis=-1).tolist()

    all_confidences.extend(confidences)
    all_preds.extend(preds)
    all_labels_list.extend(labs)

all_confidences   = np.array(all_confidences)
all_preds         = np.array(all_preds)
all_labels_array  = np.array(all_labels_list)
correct_mask      = (all_preds == all_labels_array)

test_acc = correct_mask.mean()
test_f1  = f1_metric.compute(predictions=all_preds.tolist(),
                               references=all_labels_array.tolist(),
                               average="macro")["f1"]

print(f"\nTest metrics:")
print(f"  Accuracy   : {test_acc*100:.2f}%")
print(f"  Macro F1   : {test_f1*100:.2f}%")
print(f"\nConfidence stats:")
print(f"  mean={all_confidences.mean():.3f}  std={all_confidences.std():.3f}  "
      f"min={all_confidences.min():.3f}  max={all_confidences.max():.3f}")

In [ ]:
# ── Confidence histogram: correct vs wrong ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = np.linspace(0.3, 1.0, 15)

axes[0].hist(all_confidences[correct_mask],  bins=bins, alpha=0.7,
             color="#5cb85c", label="Correct")
axes[0].hist(all_confidences[~correct_mask], bins=bins, alpha=0.7,
             color="#d9534f", label="Wrong")
axes[0].set_xlabel("Softmax confidence (max class)")
axes[0].set_ylabel("Count")
axes[0].set_title("Confidence Distribution — Correct vs Wrong", fontweight="bold")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Reliability diagram (calibration)
bin_centers, accuracy_per_bin, count_per_bin = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (all_confidences >= lo) & (all_confidences < hi)
    if mask.sum() > 0:
        bin_centers.append((lo + hi) / 2)
        accuracy_per_bin.append(correct_mask[mask].mean())
        count_per_bin.append(mask.sum())

axes[1].plot([0.3, 1.0], [0.3, 1.0], "k--", label="Perfect calibration", alpha=0.5)
axes[1].scatter(bin_centers, accuracy_per_bin, s=[c*0.3 for c in count_per_bin],
                color="#4e91d4", alpha=0.8, label="Model (bubble ∝ count)")
axes[1].set_xlabel("Mean confidence in bin")
axes[1].set_ylabel("Fraction correct")
axes[1].set_title("Reliability Diagram", fontweight="bold")
axes[1].set_xlim(0.3, 1.0); axes[1].set_ylim(0.3, 1.05)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Calibration Analysis — tweet_eval Test Set", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("calibration.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved ✓")

# Calibration commentary
gaps = [abs(c - a) for c, a in zip(bin_centers, accuracy_per_bin)]
mean_gap = np.mean(gaps) if gaps else 0
print(f"\nMean calibration gap (ECE proxy): {mean_gap:.3f}")
if mean_gap < 0.05:
    print("Model is well-calibrated ✓")
elif mean_gap < 0.10:
    print("Moderate overconfidence — consider temperature scaling")
else:
    print("Significant miscalibration — Platt scaling or temperature scaling recommended")

## Task 5 — Attention Inspection

We load the **encoder backbone** (not the classifier) with `output_attentions=True`, pass a chosen tweet, extract the last layer's attention weights, and visualise what the `[CLS]` token attends to — revealing the evidence tokens behind the prediction.

### How DistilBERT attention works
- 6 transformer layers, each with 12 heads
- Each head produces a `(seq_len, seq_len)` attention matrix
- We average across all 12 heads of the **last layer** and inspect the row corresponding to `[CLS]` (position 0) — it captures the global sentence representation used by the classifier

In [ ]:
encoder = AutoModel.from_pretrained(
    OUTPUT_DIR if os.path.exists(OUTPUT_DIR) else MODEL_NAME,
    output_attentions = True,
).to(DEVICE)
encoder.eval()

def get_cls_attention(text: str):
    """
    Returns (tokens, cls_attn_weights) for the last DistilBERT layer,
    averaged across all attention heads, for the CLS → token direction.
    """
    enc = tokenizer(text, return_tensors="pt",
                    truncation=True, max_length=MAX_LEN)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        out = encoder(**enc)

    # out.attentions: tuple of (n_layers,) each (1, n_heads, seq, seq)
    last_layer_attn = out.attentions[-1]              # (1, 12, seq, seq)
    mean_attn       = last_layer_attn.squeeze(0).mean(dim=0)   # (seq, seq)
    cls_attn        = mean_attn[0, :].cpu().numpy()            # CLS → all tokens

    token_ids = enc["input_ids"][0].cpu().tolist()
    tokens    = tokenizer.convert_ids_to_tokens(token_ids)

    # Only return non-padding tokens
    real_len  = int(enc["attention_mask"][0].sum().item())
    return tokens[:real_len], cls_attn[:real_len]


def plot_cls_attention(text: str, ax=None, title_suffix=""):
    tokens, attn = get_cls_attention(text)
    standalone   = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(max(8, len(tokens) * 0.5), 3))

    colors = plt.cm.Blues(attn / attn.max())
    bars   = ax.bar(range(len(tokens)), attn, color=colors, edgecolor="white")
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("[CLS] attention weight")
    ax.set_title(f"[CLS] → Token Attention (last layer, mean heads){title_suffix}",
                 fontweight="bold")
    ax.grid(True, alpha=0.2, axis="y")

    # Highlight top-3 tokens
    top3_idx = np.argsort(attn)[-3:]
    for idx in top3_idx:
        bars[idx].set_edgecolor("#e07b39")
        bars[idx].set_linewidth(2)
        ax.text(idx, attn[idx] + 0.003, tokens[idx],
                ha="center", fontsize=8, color="#e07b39", fontweight="bold")

    if standalone:
        plt.tight_layout()
        plt.show()
    return tokens, attn


# ── Visualise one example per label ──────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

for ax, (lbl, texts) in zip(axes, examples_by_label.items()):
    tweet = texts[0]
    color_title = {0: "🔴 Negative", 1: "🟡 Neutral", 2: "🟢 Positive"}
    tokens, attn = plot_cls_attention(
        tweet, ax=ax,
        title_suffix=f"\n\"{tweet[:80]}{'…' if len(tweet)>80 else ''}\"  [{LABEL_NAMES[lbl].upper()}]"
    )

plt.suptitle("CLS Attention Inspection — tweet_eval Examples",
             fontweight="bold", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("attention_inspection.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved ✓")

## Deliverable — `analyze_text()` Production Helper

This function is the final artifact: it returns `label`, `confidence`, and `highlighted_tokens` — everything a support engineer needs to triage a ticket or explain a decision to a stakeholder.

In [ ]:
def analyze_text(text: str, top_k: int = 5) -> dict:
    """
    Production-style inference helper.

    Returns
    -------
    dict with keys:
        label             : "negative" | "neutral" | "positive"
        confidence        : float — softmax probability of predicted class
        class_probs       : dict mapping each label to its probability
        highlighted_tokens: list of (token, weight) for the top-k evidence tokens
        escalate          : bool — True if Negative with confidence > 0.80
    """
    # ── Classification ────────────────────────────────────────────────────────
    enc    = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    enc_d  = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        logits = clf_model(**enc_d).logits           # (1, 3)
    probs    = F.softmax(logits, dim=-1)[0].cpu()   # (3,)
    pred_idx = int(probs.argmax())
    label    = LABEL_NAMES[pred_idx]
    conf     = float(probs[pred_idx])

    # ── Attention evidence ────────────────────────────────────────────────────
    tokens, attn_weights = get_cls_attention(text)

    # Skip [CLS] and [SEP] for the highlight list
    evidence = [(t, float(w)) for t, w in zip(tokens, attn_weights)
                if t not in ("[CLS]", "[SEP]", "<s>", "</s>", "[PAD]")]
    evidence.sort(key=lambda x: x[1], reverse=True)

    return {
        "label"             : label,
        "confidence"        : round(conf, 4),
        "class_probs"       : {LABEL_NAMES[i]: round(float(probs[i]), 4) for i in range(3)},
        "highlighted_tokens": evidence[:top_k],
        "escalate"          : label == "negative" and conf > 0.80,
    }


# ── Demo on representative support messages ───────────────────────────────────
support_messages = [
    "Your product is absolutely terrible and I want a full refund NOW.",
    "The delivery arrived on time and the package was fine.",
    "I had a great experience, the support agent was incredibly helpful!",
    "Not sure how I feel about this, it was okay I guess.",
    "THREE DAYS and still no response. This is the worst customer service I've ever seen.",
]

print("=== analyze_text() — Production Inference Demo ===\n")
for msg in support_messages:
    result = analyze_text(msg)
    print(f"Text       : {msg[:80]}{'…' if len(msg)>80 else ''}")
    print(f"Prediction : {result['label'].upper()}  (confidence={result['confidence']:.3f})")
    print(f"Class probs: {result['class_probs']}")
    print(f"Top tokens : {[(t, round(w, 4)) for t, w in result['highlighted_tokens'][:3]]}")
    if result["escalate"]:
        print(f"⚠  ESCALATE — high-confidence negative sentiment")
    print("─" * 70)

In [ ]:
# ── Visualise evidence tokens for one negative example ───────────────────────
neg_text = "Your product is absolutely terrible and I want a full refund NOW."
result   = analyze_text(neg_text)

fig, ax = plt.subplots(figsize=(12, 3.5))
plot_cls_attention(
    neg_text, ax=ax,
    title_suffix=f"\nPrediction: {result['label'].upper()}  (conf={result['confidence']:.3f})"
)
plt.tight_layout()
plt.savefig("attention_negative_example.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\nTop-5 evidence tokens for this prediction:")
for tok, w in result["highlighted_tokens"]:
    bar = "█" * int(w * 500)
    print(f"  {tok:<15} {w:.4f}  {bar}")

print("\nInsight: the model attends most strongly to emotionally charged words")
print("('terrible', 'refund', 'want') when predicting NEGATIVE sentiment.")

## Evaluation Report & Deployment Summary

### Final Metrics

| Metric | Validation | Test |
|---|---|---|
| **Accuracy** | *(see above)* | *(see above)* |
| **Macro F1** | *(see above)* | *(see above)* |
| **Best epoch** | Epoch with highest val F1 | — |

### Saved Artifacts

| File | Purpose |
|---|---|
| `distilbert_tweet_eval/` | Fine-tuned model weights + config |
| `distilbert_tweet_eval/tokenizer_config.json` | Tokenizer for inference reproducibility |
| `class_distribution.png` | Dataset class balance per split |
| `training_curves.png` | Loss / accuracy / F1 per epoch |
| `calibration.png` | Confidence histogram + reliability diagram |
| `attention_inspection.png` | CLS attention per label |
| `attention_negative_example.png` | Evidence token spotlight |

### Deployment Checklist

- [x] Confidence threshold: route `confidence < 0.70` to human queue
- [x] Escalation flag: `escalate=True` if negative + conf > 0.80
- [x] Evidence tokens surfaced — every decision is explainable to a support agent
- [ ] Add PII redaction before passing live tickets to the model
- [ ] Monitor accuracy weekly via `evaluate` on a random sample of re-labeled tickets
- [ ] Consider `xlm-roberta-base` for multilingual support channels